# CondAptNet — Stage 1 Training (Colab / T4)

**Before running:** Set runtime to GPU (Runtime → Change runtime type → T4 GPU).  
**Steps:** GPU check → clone repo → download data → resume check → train → evaluate.

In [ ]:
# Cell 1 — GPU check
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → GPU (T4)."
)

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024 ** 3
print(f"GPU  : {gpu.name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
assert vram_gb >= 12, f"Expected ≥12 GB VRAM, got {vram_gb:.1f} GB"
print("GPU check passed.")

In [ ]:
# Cell 2 — Clone repo and install dependencies
import subprocess, sys

result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/shivanshb828/CondAptNet.git"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Repo cloned.")
else:
    # Already cloned — pull latest
    subprocess.run(["git", "-C", "CondAptNet", "pull", "--quiet"],
                   check=True)
    print("Repo already present — pulled latest.")

import os
os.chdir("CondAptNet")
print(f"Working directory: {os.getcwd()}")

packages = [
    "fair-esm",
    "ViennaRNA",
    "scikit-learn",
    "scipy",
    "biopython",
    "gdown",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    check=True
)
print("Dependencies installed.")

In [ ]:
# Cell 3 — Download data from Google Drive
# Replace the IDs below with your actual Google Drive file/folder IDs.
# master_dataset.csv  → share as Anyone with link, copy the file ID
# vienna_cache.pkl    → same
# protein_embeddings  → share the folder, copy the folder ID

import os
import gdown

MASTER_DATASET_ID      = "1LlVsXnMkrEZm-JlkVQv8Bwffxk8hueqB"
VIENNA_CACHE_ID        = "1mRaGNcVsaSb7QywORqCHSIMQWcHXpvV2"
PROTEIN_EMBEDDINGS_ID  = "1O_y08uU7Gs14LIKBCZ02L1QqRIItmvuQ"

os.makedirs("data/processed/protein_embeddings", exist_ok=True)
os.makedirs("models/checkpoints/pretrain",       exist_ok=True)

print("Downloading master_dataset.csv ...")
gdown.download(
    f"https://drive.google.com/uc?id={MASTER_DATASET_ID}",
    "data/processed/master_dataset.csv",
    quiet=False,
)

print("Downloading vienna_cache.pkl ...")
gdown.download(
    f"https://drive.google.com/uc?id={VIENNA_CACHE_ID}",
    "data/processed/vienna_cache.pkl",
    quiet=False,
)

print("Downloading protein_embeddings folder ...")
gdown.download_folder(
    f"https://drive.google.com/drive/folders/{PROTEIN_EMBEDDINGS_ID}",
    output="data/processed/protein_embeddings",
    quiet=False,
    use_cookies=False,
)

# Verify
import pandas as pd
df = pd.read_csv("data/processed/master_dataset.csv")
ready = (
    df["sequence"].notna() &
    (df["needs_sequence_enrichment"] == False) &
    df["protein_sequence"].notna()
).sum()
n_emb = len(os.listdir("data/processed/protein_embeddings"))
print(f"master_dataset.csv : {len(df):,} rows  ({ready:,} training-ready)")
print(f"protein_embeddings : {n_emb} .npy files")
print("Data download complete.")

In [ ]:
# Cell 4 — Resume logic
# Finds the latest epoch checkpoint so training can continue after a disconnect.

import glob

CHECKPOINT_DIR = "models/checkpoints/pretrain"

existing = sorted(glob.glob(f"{CHECKPOINT_DIR}/epoch_*.pt"))

if existing:
    latest = existing[-1]
    import torch
    meta = torch.load(latest, map_location="cpu")
    last_epoch    = meta.get("epoch", "?")
    best_val_mcc  = meta.get("best_val_mcc", meta.get("val_mcc", float("nan")))
    patience      = meta.get("patience_count", "?")
    RESUME_FLAG   = "--resume"
    print(f"Checkpoint found  : {latest}")
    print(f"Last epoch        : {last_epoch}")
    print(f"Best val MCC      : {best_val_mcc:.4f}")
    print(f"Patience count    : {patience}")
    print("Will resume training from next epoch.")
else:
    RESUME_FLAG = ""
    print("No checkpoint found — starting fresh.")

In [ ]:
# Cell 5 — Train (T4 settings: batch_size=16, max_prot_len=128)
import subprocess, os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "python", "scripts/training/train.py",
    "--max-epochs",    "100",
    "--batch-size",    "16",
    "--max-prot-len",  "128",
    "--checkpoint-dir", CHECKPOINT_DIR,
]
if RESUME_FLAG:
    cmd.append(RESUME_FLAG)

env = os.environ.copy()
env["PYTHONUNBUFFERED"]    = "1"
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 60)
print(f"Training finished (exit code {proc.returncode}).")

In [ ]:
# Cell 6 — Evaluate best checkpoint
import subprocess, os

best_ckpt = os.path.join(CHECKPOINT_DIR, "best.pt")

if not os.path.exists(best_ckpt):
    print(f"No best checkpoint at {best_ckpt} — run Cell 5 first.")
else:
    for split in ("val", "test"):
        print(f"\n{'='*55}")
        print(f"Evaluating on {split.upper()} split")
        print(f"{'='*55}")
        result = subprocess.run(
            [
                "python", "scripts/evaluation/evaluate.py",
                "--checkpoint", best_ckpt,
                "--split",      split,
            ],
            capture_output=False, text=True,
        )